In [ ]:
# Standard libraries
import numpy as np
from pathlib import Path
from glob import glob
import plotly.express as px
import matplotlib.pyplot as plt
from tqdm import tqdm
from functools import partial
import pandas as pd
import ants
import seaborn as sns

# Specific libraries
import nibabel as nib
from scipy.ndimage import distance_transform_edt
import colorcet

# custom functions
import registration as reg
import calib_utils as cal

# Ghost libraries
from ghost.phantom import Caliber137

In [ ]:
calib_matrix_dir = Path(f"./calibration/calib")

In [ ]:
parent_dir = Path("./caliber_phantom")
eval_dir = parent_dir / "derivatives"
plot_dir = parent_dir / "plots"
plot_dir.mkdir(exist_ok=True)
testname = eval_dir.parent.stem

eval_params_fname = eval_dir / (testname + "_" + "params.h5")
eval_scan_fname = eval_dir / (testname + "_" + "RAW_REG.nii.gz")
eval_mask_fname = eval_dir / (testname + "_" + "MASK.nii.gz")

# Load in sequence parameters
test_df = pd.read_hdf(eval_params_fname, key="df", mode="r")

# Load in scan data and add into data frame
test_nii = nib.load(eval_scan_fname)
test_data = test_nii.get_fdata()
test_aff = test_nii.affine
test_spacing = reg.spacing_from_affine(test_aff)

test_data_reshuffled = np.transpose(test_data,[3,0,1,2]).astype(np.float32)
test_df["image"] = list(test_data_reshuffled)

# Load in mask
eval_mask_nii = nib.load(eval_mask_fname)
eval_mask = eval_mask_nii.get_fdata().astype(bool)

In [ ]:
test_df.sort_values("scan_time", inplace=True, ignore_index=True)

# Put the b=0 image on b=0 rows, NaN elsewhere
b0_threshold = 5
test_df["s0_image"] = test_df["image"].where(test_df["b_value"] < b0_threshold)

# Forward-fill so each DWI scan gets the most recent preceding b=0 image
test_df["s0_image"] = test_df["s0_image"].ffill()

# Keep only diffusion-weighted scans
dwi_df = test_df[test_df["b_value"] > b0_threshold].copy()

# Optional safety check: drop rows that do not have a preceding b=0 scan
dwi_df = dwi_df[dwi_df["s0_image"].notna()].copy()

In [ ]:
# Prepare empty columns
dwi_df["mask"] = None
dwi_df["spacing"] = None
dwi_df["pred_bmap"] = None

for idx, row in dwi_df.iterrows():
    dur = row["dwi_grad_duration"]
    
    # Build the path to the calibration file
    A_path = calib_matrix_dir / f"calibration_A_DUR_{int(dur*1e3):.0f}.nii.gz"
    
    # Unpack calibration data
    A, mask, sigma, spacing = cal.unpack_A_nii(A_path)
    
    # Compute pred_bmap
    pred_bmap = cal.est_bmap(
        row["grad_vec"],
        delta_small=dur,
        delta_big=row["TE"] / 2,
        A=A,
        mask=mask,
        sigma=sigma
    )
    
    # Store results
    dwi_df.at[idx, "mask"] = mask
    dwi_df.at[idx, "spacing"] = spacing
    dwi_df.at[idx, "pred_bmap"] = pred_bmap

In [ ]:
n_scans = len(dwi_df)

s_scans_list = []
s0_scans_list = []
pred_bmaps_resampled_list = []
masks_resampled_list = []

for i, row in dwi_df.iterrows():
    # Extract per-row data
    s = row["image"]
    s0 = row["s0_image"]
    pred_bmap = row["pred_bmap"]
    mask = row["mask"]
    spacing = row["spacing"]

    # Build resampler for THIS row
    resample_func = partial(
        reg.resample_array,
        current_spacing=spacing,
        target_spacing=test_spacing,
        interp_type=0
    )

    # Resample bmap and mask
    pred_bmap_resampled = resample_func(pred_bmap)
    mask_resampled = resample_func(mask)

    # Store
    s_scans_list.append(s)
    s0_scans_list.append(s0)
    pred_bmaps_resampled_list.append(pred_bmap_resampled)
    masks_resampled_list.append(mask_resampled)

# Stack original scans
s_scans = np.stack(s_scans_list, axis=-1)
s0_scans = np.stack(s0_scans_list, axis=-1)
pred_bmaps_resampled = np.stack(pred_bmaps_resampled_list, axis=-1)
masks_resampled = np.stack(masks_resampled_list, axis=-1)

b_values = dwi_df["b_value"].values
b_values_reshaped = b_values.reshape((1, 1, 1, -1))
dw_axes = dwi_df["dwi_direction"].values

# --- Crop to common overlap ---
overlap_shape = [
    min(a, b)
    for a, b in zip(pred_bmaps_resampled[..., -1].shape, s_scans[..., 0].shape)
]

s_scans_cropped = np.stack(
    [cal.center_crop_3d(s_scans[..., i], overlap_shape) for i in range(n_scans)],
    axis=-1
)

s0_scans_cropped = np.stack(
    [cal.center_crop_3d(s0_scans[..., i], overlap_shape) for i in range(n_scans)],
    axis=-1
)

pred_bmaps_cropped = np.stack(
    [cal.center_crop_3d(pred_bmaps_resampled[..., i], overlap_shape) for i in range(n_scans)],
    axis=-1
)

masks_cropped = np.stack(
    [cal.center_crop_3d(masks_resampled[..., i], overlap_shape) for i in range(n_scans)],
    axis=-1
)

# --- Build valid mask (now per scan) ---
valid_mask = (eval_mask[..., None] & masks_cropped.astype(bool))

# --- ADC calculations ---
adc_trad = np.zeros_like(s_scans_cropped)
adc_trad[valid_mask] = np.nan_to_num(
    np.log(s0_scans_cropped[valid_mask] / s_scans_cropped[valid_mask]) / b_values.repeat(np.prod(overlap_shape)).reshape(valid_mask.shape)[valid_mask],
    nan=0, posinf=0, neginf=0
)

adc_calib = np.zeros_like(s_scans_cropped)
adc_calib[valid_mask] = np.nan_to_num(
    np.log(s0_scans_cropped[valid_mask] / s_scans_cropped[valid_mask]) / pred_bmaps_cropped[valid_mask],
    nan=0, posinf=0, neginf=0
)

# --- Average repeats ---
adc_trad_avg = np.zeros([*adc_trad.shape[:-1], 3])
adc_calib_avg = np.zeros([*adc_trad.shape[:-1], 3])

for i, a in enumerate(np.unique(dw_axes)):
    rep_mask = dw_axes == a
    adc_trad_avg[..., i] = np.mean(adc_trad[..., rep_mask], axis=-1)
    adc_calib_avg[..., i] = np.mean(adc_calib[..., rep_mask], axis=-1)

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(8, 8), facecolor="k")

colormap = colorcet.cm.rainbow_bgyr_10_90_c83
i_slice = 22
valid_mask_3d = np.all(valid_mask, axis=-1)

vmin=0.5
vmax=2.5
dif_lim = 0.4

fs_title = 15
fs_cbar = 10

labels = [rf"$ADC_{a}$" for a in np.unique(dw_axes)]

row_mappables = [None, None, None]
for i in range(3):
    trad_slice = adc_trad_avg[..., i_slice, i]
    calib_slice = adc_calib_avg[..., i_slice, i]
    trad_slice = np.rot90(trad_slice, k=1, axes=(0, 1))[7:-7, 6:-7]
    calib_slice = np.rot90(calib_slice, k=1, axes=(0, 1))[7:-7, 6:-7]

    trad_slice_m = trad_slice * 1e3
    calib_slice_m = calib_slice * 1e3

    mask_slice = np.rot90(valid_mask_3d[:, :, i_slice], k=1, axes=(0, 1))[7:-7, 6:-7]
    plot_mask = np.full_like(trad_slice, np.nan, dtype=float)
    plot_mask[mask_slice == 0] = 1

    im0 = axes[0, i].imshow(trad_slice_m, cmap=colormap, vmin=vmin, vmax=vmax)
    axes[0, i].imshow(plot_mask, cmap="Greys", vmin=0, vmax=1)
    im1 = axes[1, i].imshow(calib_slice_m, cmap=colormap, vmin=vmin, vmax=vmax)
    axes[1, i].imshow(plot_mask, cmap="Greys", vmin=0, vmax=1)
    im2 = axes[2, i].imshow(trad_slice_m - calib_slice_m, cmap=colorcet.cm.coolwarm, vmin=-dif_lim, vmax=dif_lim)
    axes[2, i].imshow(plot_mask, cmap="Greys", vmin=0, vmax=1)

    if i == 0:
        row_mappables[0] = im0
        row_mappables[1] = im1
        row_mappables[2] = im2
    axes[0, i].set_title(dw_axes[i], fontsize=fs_title, style="italic", color="white")

for ax in axes.flatten():
    ax.set_axis_off()

plt.tight_layout(rect=[0.15, 0, 0.95, 0.95])

# Add row labels on the left
row_labels = ["Uncalibrated", "Calibrated", "Difference"]
x_offset = -0.05

for r, label in enumerate(row_labels):
    # Get position of the leftmost subplot in this row
    pos = axes[r, 0].get_position()
    
    fig.text(
        pos.x0 - 0.015,                    # move slightly left of plots
        pos.y0 + pos.height / 2,           # vertically centered on row
        label,
        va="center",
        ha="right",
        fontsize=fs_title,
        rotation=90,
        style="italic",
        color="white"
    )

cbar_width = 0.01
cbar_pad = 0.05

for r in range(3):

    pos_left = axes[r, 0].get_position()
    pos_right = axes[r, -1].get_position()

    # vertical span of the row
    shrink = 0.7
    full_height = pos_left.height
    y0 = pos_left.y0 + full_height * (1-shrink)/ 2

    # position to the right of last column
    x0 = pos_right.x1 + cbar_pad

    cax = fig.add_axes([x0, y0, cbar_width, full_height * shrink])
    cbar = fig.colorbar(row_mappables[r], cax=cax)

    cbar.ax.tick_params(colors="white", labelsize=10)
    cbar.outline.set_edgecolor("white")

    if r == 2:
        cbar.ax.set_title("$\Delta\ ADC$ \n$( \\times 10^{-3}\ mm^2/s)$", color="white", pad=10, fontsize=fs_cbar, loc="center")
    else:
        cbar.ax.set_title("$ADC$ \n$( \\times 10^{-3}\ mm^2/s)$", color="white", pad=10, fontsize=fs_cbar, loc="center")
    
fmt = "png"
plot_fname = plot_dir / (testname + "_" + f"adc_map_comparison.{fmt}")
fig.savefig(fname=plot_fname, dpi=600, format=fmt, bbox_inches="tight", pad_inches=0.25)

In [ ]:
# Extract an avergae S0 image
s0_avg = test_df[test_df["b_value"]<b0_threshold]["image"].mean()
s0_a = ants.from_numpy(s0_avg, spacing=test_spacing)
# s0_a.plot_ortho(resample=True, flat=True)

# Register segmentations of different subvolumes to S0 image
phantom = Caliber137()
inv, fwd = phantom.reg_to_phantom(s0_a, do_syn=True, weighting='T2w',)

# Extract a register mask of the different sub-volumes in the Caliber Phantom
masks = np.zeros([*s0_avg.shape, 3], dtype=np.float32)
for i, weight in enumerate(["ADC", "T1mimics", "T2mimics"]):
    seg_a = phantom.warp_seg(s0_a, seg=weight, xfm=inv)
    masks[..., i] = seg_a.numpy()

    # Can plot each segmentation for confirmation of alignment
    # s0_a.plot(axis=2, overlay=seg_a, slices=np.arange(15, 35))

In [ ]:
# Combine sub-volume masks into single 'non-background' mask
combined_masks = np.any(masks, axis=3)

# Extract mask for only the flood filled regions of the phantom
background_mask = (valid_mask_3d) & (~combined_masks)

# Plot for confirmation
plot_data = np.concatenate([combined_masks, valid_mask_3d, background_mask], axis=1)
px.imshow(plot_data, animation_frame=2, height=800)

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(16,8))

for i, dataset in enumerate([adc_trad_avg, adc_calib_avg]):

    for j in range(3):
        
        bg_vals = dataset[..., j][background_mask] * 1e3

        bg_median = np.median(bg_vals)
        lq = np.percentile(bg_vals, 25)
        uq = np.percentile(bg_vals, 75)
        sigma = np.std(bg_vals)

        label = ["Trad", "Calib"][i] + "_" + "XYZ"[j]
        sns.kdeplot(bg_vals, clip=(1,3), color=[0, 0.2+(1-i)*j/3, 0.2+i*j/3], label=label)
        print(label)
        print(f"Median: {bg_median:.4f}, LQ: {lq:.4f}, UQ:{uq:.4f}, IQR:{(uq-lq):.4f}, std: {sigma:.4f}\n")

ax.legend()
ax.set_xlabel("ADC")
fig.show()
